In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "DTD_Replication_Study" for domain "drug" and was generated for All of Us Controlled Tier Dataset v8
dataset_93744883_drug_sql <- paste("
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_exposure.stop_reason,
        d_exposure.refills,
        d_exposure.quantity,
        d_exposure.days_supply,
        d_exposure.sig,
        d_exposure.route_concept_id,
        d_route.concept_name as route_concept_name,
        d_exposure.lot_number,
        d_exposure.visit_occurrence_id,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value,
        d_exposure.dose_unit_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `cb_criteria` cr             
                        WHERE
                            concept_id IN (21604490, 21604557, 21604686, 21604729, 21604788)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `cb_criteria` cr             
                                    WHERE
                                        concept_id IN (21604686, 21604788, 21604729)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) d_exposure 
        LEFT JOIN
            `concept` d_standard_concept 
                ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
        LEFT JOIN
            `concept` d_type 
                ON d_exposure.drug_type_concept_id = d_type.concept_id 
        LEFT JOIN
            `concept` d_route 
                ON d_exposure.route_concept_id = d_route.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` d_visit 
                ON v.visit_concept_id = d_visit.concept_id 
        LEFT JOIN
            `concept` d_source_concept 
                ON d_exposure.drug_source_concept_id = d_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
drug_93744883_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "drug_93744883",
  "drug_93744883_*.csv")
message(str_glue('The data will be written to {drug_93744883_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_93744883_drug_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  drug_93744883_path,
  destination_format = "CSV")

In [ ]:
#– Read in drug data 
drug_25694592_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20251014", 
  "drug_25694592",
  "drug_25694592_*.csv")
message(str_glue('The data will be written to {drug_25694592_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))


In [ ]:
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), drug_type_concept_name = col_character(), stop_reason = col_character(), sig = col_character(), route_concept_name = col_character(), lot_number = col_character(), visit_occurrence_concept_name = col_character(), drug_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), route_source_value = col_character(), dose_unit_source_value = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_25694592_drug_df <- read_bq_export_from_workspace_bucket(drug_25694592_path)

In [ ]:
# Load required libraries
library(dplyr)
library(stringr)

medication_df <- dataset_25694592_drug_df

#--- Extract antideps
known_drugs <- c('citalopram', 'escitalopram', 'fluoxetine', 'fluvoxamine', 'paroxetine', 'sertraline',
                'desvenlafaxine', 'duloxetine', 'levomilnacipran', 'milnacipran', '(milnacipran', 'venlafaxine',
                'bupropion', 'mirtazapine',
                'nefazodone', '(nefazodone', 'trazodone', '(vilazodone', 'vilazodone', 'vortioxetine', 'viloxazine',
                'imipramine', 'amitriptyline', 'amoxapine', 'clomipramine', 'desipramine', 'doxepin',
                'maprotiline', 'nortriptyline', 'protriptyline', 'trimipramine',
                'isocarboxazid', 'phenelzine', 'selegiline', 'tranylcypromine',
                '5-hydroxytryptophan', 'esketamine', '(esketamine')

In [ ]:
# Function to extract drug name from any position in the split string
extract_name <- function(concept_name) {
  # Split the string and convert to lowercase
  words <- tolower(strsplit(concept_name, " ", fixed = TRUE)[[1]])
  
  # Find which word matches our known drugs
  match_idx <- which(words %in% known_drugs)
  
  # Return the first match, or NA if no match found
  if (length(match_idx) > 0) {
    return(words[match_idx[1]])
  } else {
    return(NA_character_)
  }
}

In [ ]:
#-- Apply the function to extract drug names
medication_df$drug <- sapply(medication_df$standard_concept_name, extract_name)

#-- Filter out rows where no antidepressant was found (NA values)
antidep_df <- medication_df[!is.na(medication_df$drug), ]

#-- Antipsychotic df (where no antidepressant was found)
antipsychotic_df <- medication_df[is.na(medication_df$drug), ]

#-- Clean antidepressant names
antidep_df <- antidep_df %>%
  mutate(
    drug = case_when(
      drug == "(milnacipran" ~ "milnacipran",
      drug == "(nefazodone" ~ "nefazodone",
      drug == "(vilazodone" ~ "vilazodone",
      drug == "(esketamine" ~ "esketamine",
      TRUE ~ drug
    )
  )


In [ ]:
# Mechanism-based classification
antidep_df$Category <- case_when(
  antidep_df$drug %in% c('citalopram', 'escitalopram', 'fluoxetine', 'fluvoxamine', 'paroxetine', 'sertraline') ~ 'SSRI',
  antidep_df$drug %in% c('desvenlafaxine', 'duloxetine', 'levomilnacipran', 'milnacipran', 'venlafaxine') ~ 'SNRI',
  antidep_df$drug %in% c('viloxazine', 'maprotiline') ~ 'NRI',
  antidep_df$drug %in% c('bupropion') ~ 'NDRI',
  antidep_df$drug %in% c('mirtazapine','mianserin') ~ 'NaSSA',
  antidep_df$drug %in% c('trazodone','nefazodone') ~ 'SARI',
  antidep_df$drug %in% c('vortioxetine','vilazodone') ~ 'Serotonin_Modulator',
  antidep_df$drug %in% c('5-hydroxytryptophan','oxitriptan') ~ 'Serotonin_Precursor',
  antidep_df$drug %in% c('imipramine','amitriptyline','clomipramine','desipramine','doxepin','nortriptyline','protriptyline','trimipramine', 'amoxapine') ~ 'TCA',
  antidep_df$drug %in% c('isocarboxazid', 'phenelzine', 'selegiline', 'tranylcypromine') ~ 'MAOI',
  antidep_df$drug %in% c('esketamine') ~ 'NMDA_Antagonist',
  antidep_df$drug %in% c('moclobemide') ~ 'RIMA',
  TRUE ~ NA_character_
)

In [ ]:
#-- Extract second generation antipsychotics
known_second_gen_antipsychotics <- c("olanzapine", "risperidone", "quetiapine", "(quetiapine", "aripiprazole", "ziprasidone", "clozapine",
                          "paliperidone", "asenapine", "lurasidone", "iloperidone", "cariprazine", "brexpiprazole", 
                          "lumateperone", "amisulpride", "sulpiride", "pimavanserin", "(cariprazine", "(iloperidone")


# Function to extract drug name from first position in the split string
extract_second_gen_antipsychotics_name <- function(concept_name) {
  # Split the string and convert to lowercase
  words <- tolower(strsplit(concept_name, " ", fixed = TRUE)[[1]])
  
  # Find which word matches our known_second_gen_antipsychotics
  match_idx <- which(words %in% known_second_gen_antipsychotics)
  
  # Return the first match, or NA if no match found
  if (length(match_idx) > 0) {
    return(words[match_idx[1]])
  } else {
    return(NA_character_)
  }
}


#-- Apply the function to extract second-gen antipsychotic drug names
antipsychotic_df$drug <- sapply(antipsychotic_df$standard_concept_name, extract_second_gen_antipsychotics_name)
atypical_antipsychotic_df <-  antipsychotic_df[!is.na(antipsychotic_df$drug), ]
atypical_antipsychotic_df$Category <- "Atypical Antipsychotic"


In [ ]:
#-- Clean names
atypical_antipsychotic_df <- atypical_antipsychotic_df %>%
  mutate(
    drug = case_when(
      drug == "(quetiapine" ~ "quetiapine",
      drug == "(cariprazine" ~ "cariprazine",
      drug == "(iloperidone" ~ "iloperidone",
      TRUE ~ drug
    )
  )

#-- extract first generation antipsychotics
known_first_gen_antipsychotics <- c("haloperidol", "chlorpromazine", "prochlorperazine", "fluphenazine", "perphenazine", "trifluoperazine", 
                                    "thioridazine", "thiothixene", "loxapine", "mesoridazine", "pimozide", "droperidol", "molindone", "promazine")

# Function to extract drug name from any position in the split string
extract_first_gen_antipsychotics_name <- function(concept_name) {
  # Split the string and convert to lowercase
  words <- tolower(strsplit(concept_name, " ", fixed = TRUE)[[1]])
  
  # Find which word matches our known_first_gen_antipsychotics
  match_idx <- which(words %in% known_first_gen_antipsychotics)
  
  # Return the first match, or NA if no match found
  if (length(match_idx) > 0) {
    return(words[match_idx[1]])
  } else {
    return(NA_character_)
  }
}

#-- Typical antipsychotic df
typical_antipsychotic_df <- antipsychotic_df[is.na(antipsychotic_df$drug), ]
typical_antipsychotic_df$drug <- sapply(typical_antipsychotic_df$standard_concept_name, extract_first_gen_antipsychotics_name)


In [ ]:
#-- typical antipsychotic categories
typical_antipsychotic_df <- typical_antipsychotic_df %>%
  mutate(
    drug = case_when(
      standard_concept_name == "lithium" ~ "lithium",
      TRUE ~ drug
    )
  ) %>%
  mutate(
    Category = case_when(
      drug == "lithium" ~ "Mood_Stabilizer",
      !is.na(drug) ~ "Typical Antipsychotic",
      TRUE ~ NA
    )
  )

#-- Everything else is an amino acid (precursors for serotonin)
amino_acid_df <- typical_antipsychotic_df[is.na(typical_antipsychotic_df$drug), ]
amino_acid_df <- amino_acid_df %>%
  mutate(
    drug = "AminoAcid",
    Category = "AminoAcid"
  )
typical_antipsychotic_df <-  typical_antipsychotic_df[!is.na(typical_antipsychotic_df$drug), ]


In [ ]:
#-- Function to augmentation therapy from matches in the split string
extract_augmentation <- function(concept_name) {
  # Split the string and convert to lowercase
  words <- tolower(strsplit(concept_name, " ", fixed = TRUE)[[1]])
  
  # Find which word matches our known drug lists
  antidep_match_idx <- which(words %in% known_drugs)
  second_gen_antipsych_match_idx <- which(words %in% known_second_gen_antipsychotics)
  first_gen_antipsych_match_idx <- which(words %in% known_first_gen_antipsychotics)
  lithium_match_idx <- which(words == "lithium")
  
  # Return the first match, or NA if no match found
  if (length(second_gen_antipsych_match_idx) > 0) {
    return(words[second_gen_antipsych_match_idx[1]])
  } else if (length(first_gen_antipsych_match_idx) > 0) {
    return(words[first_gen_antipsych_match_idx[1]])
    } else {
    return(NA_character_)
  }
}

#-- Find augmentation strategies
antidep_df$augmentation_drug <- sapply(antidep_df$standard_concept_name, extract_augmentation)
antidep_df <- antidep_df %>%
  mutate(
    Category = case_when(
      !is.na(augmentation_drug) ~ "Augmentation",
      TRUE ~ Category
    ),
    drug = case_when(
      !is.na(augmentation_drug) ~ paste0(drug, "_", augmentation_drug),
      TRUE ~ drug
    )
  )


In [ ]:
#--- Combine all scripts across categories
scripts <- bind_rows(antidep_df, atypical_antipsychotic_df, typical_antipsychotic_df, amino_acid_df)

#--- Rename drugs with category infront and select certain columns
scripts <- scripts %>%
  select(person_id, drug, Category, drug_exposure_start_datetime, drug_exposure_end_datetime, days_supply, refills) %>%
  mutate(
    drug = paste0(Category, ":", drug)
  )


In [ ]:
#-- Estimate end dat using days_supply when end_date is missing
prescriptions_EED <- scripts %>%
  mutate(
    drug_exposure_start_date = as.Date(drug_exposure_start_datetime),
    drug_exposure_end_date = as.Date(drug_exposure_end_datetime),
    estimated_end_date = case_when(
      !is.na(drug_exposure_end_date) ~ drug_exposure_end_date,
      !is.na(days_supply) ~ drug_exposure_start_date + days(days_supply),
      TRUE ~ NA_Date_
    ),
    # Flag whether end date was estimated
    end_date_estimated = is.na(drug_exposure_end_date) & !is.na(days_supply)
  ) %>%
  select(-drug_exposure_start_datetime, -drug_exposure_end_datetime)


In [ ]:
#-- 5% of scripts have missing dosage information (impute mssing as 30-days duration as it is the median for AD)
prescriptions_EED_imputed <- prescriptions_EED %>%
  mutate(
    final_end_date = case_when(
      !is.na(estimated_end_date) ~ estimated_end_date,
      is.na(estimated_end_date) ~ drug_exposure_start_date + days(30),
      TRUE ~ estimated_end_date
    ),
    imputation_flag = is.na(estimated_end_date)
  )

#-- Remove scripts that have a negative duration
prescriptions_EED_imputed_cleaned <- prescriptions_EED_imputed %>%
  select(person_id, drug, Category, drug_exposure_start_date, final_end_date) %>%
  mutate(
    PrescriptionDuration = as.numeric(difftime(final_end_date, drug_exposure_start_date, units = "days"))
  ) %>%
  filter(!PrescriptionDuration < 0) %>%
  select(-PrescriptionDuration)


In [ ]:
#-- Rank antidepressant prescriptions by Total Dispenses by year
rank_prescriptions <- prescriptions_EED_imputed_cleaned %>%
  select(person_id, drug, drug_exposure_start_date) %>%
  group_by(year = format(as.Date(drug_exposure_start_date), "%Y"), drug) %>%
  summarise(
    TotalPrescriptions = n()
  ) %>%
  group_by(year) %>%
  mutate(rank = as.integer(min_rank(-TotalPrescriptions))) %>%
  arrange(year, rank)

In [ ]:
#-- save scripts table
write.csv(prescriptions_EED_imputed_cleaned, "data/PrescriptionData.csv", row.names = FALSE)

#-- Copy to workspace bucket
system(paste(
  "gsutil -u", Sys.getenv("GOOGLE_PROJECT"), "cp data/PrescriptionData.csv",
  file.path(Sys.getenv("WORKSPACE_BUCKET"), "results/PrescriptionData.csv")
))
